In [1]:
import os
# os.environ["HF_HOME"] = "/home/seungwoochoi/data/huggingface/cache"
from tqdm import tqdm
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from FlagEmbedding import BGEM3FlagModel
import numpy as np
device = "mps"
np.set_printoptions(threshold=np.inf)
import matplotlib.pyplot as plt
import random

/opt/miniconda3/envs/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model = BGEM3FlagModel('BAAI/bge-m3', devices=device)

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 207982.02it/s]


In [3]:
import pandas as pd 
homonym_df = pd.read_csv("data/homonym_synonym.csv")
homonym_df = homonym_df.sample(frac=1).reset_index(drop=True)

In [4]:
mean1 = homonym_df['homonym'] + "(" + homonym_df['meaning1'] + ")"
mean2 = homonym_df['homonym'] + "(" + homonym_df['meaning2'] + ")"
synonym = homonym_df['synonym']

In [5]:
mean1_emb = embedding_model.encode(list(mean1))['dense_vecs'] 
mean2_emb = embedding_model.encode(list(mean2))['dense_vecs']
synonym_emb = embedding_model.encode(list(synonym))['dense_vecs']

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [12]:
surface_hadamard = mean1_emb * mean2_emb
mean_hadamard = mean1_emb * synonym_emb

surface_same = np.mean(np.sum(surface_hadamard, axis=1))
mean_same = np.mean(np.sum(mean_hadamard, axis=1))
print(surface_same)
print(mean_same)


semantic_dims = set()
random_dims_set = set()
threshold = 0.015
for i in range(mean_hadamard.shape[0]):
    dims = np.where((mean_hadamard - surface_hadamard)[i] >= threshold )[0]

    semantic_dims = semantic_dims.union(set(dims))
    random_dims_set = random_dims_set.union(set(random.sample(range(1024), 1)))
print(len(semantic_dims))
print(len(random_dims_set))



0.723
0.522
2
210


In [13]:
def unit_vector_normalize(arr):
    """
    Normalizes a NumPy array (vector) to have a magnitude (L2 norm) of 1.

    This is also known as creating a unit vector. It's useful when you
    care about the direction of the data, not the magnitude.

    Args:
        arr (np.ndarray): The input NumPy array (vector).

    Returns:
        np.ndarray: The array scaled to a magnitude of 1, or a zero vector
                    if the input vector's magnitude is zero.
    """
    # Ensure the input is a numpy array
    arr = np.asarray(arr)

    # Calculate the L2 norm (magnitude) of the vector
    norm = np.linalg.norm(arr)

    # Handle the case where the norm is zero to avoid division by zero
    if norm == 0:
        return np.zeros(arr.shape, dtype=arr.dtype)

    return arr / norm

In [8]:
# Create the dataset of tuples and save as CSV and JSONL, and display as a table.

data = [
(1, "I need your love", "I need your money", "Your affection is essential to me"),
(2, "She broke the glass", "She broke the record", "The cup shattered"),
(3, "He runs every morning", "He runs a company", "He jogs at dawn"),
(4, "The bank is closed", "The bank is flooded", "All branches are shut today"),
(5, "I saw a bat", "I saw a rat", "A winged mammal flew by"),
(6, "He charged the battery", "He charged the customer", "He powered it up"),
(7, "She left the note", "She left the team", "She placed a message on the desk"),
(8, "They booked a table", "They booked a suspect", "They made a reservation"),
(9, "The match was lit", "The match was tense", "A small stick was ignited"),
(10, "He watched the game", "He watched the gate", "He viewed the match"),
(11, "I missed the bus", "I missed the deadline", "The transport departed before I arrived"),
(12, "She opened the file", "She opened the store", "She accessed the document"),
(13, "He caught a cold", "He caught a fish", "He became ill"),
(14, "We need more time", "We need more chairs", "An extension is required"),
(15, "The seal was broken", "The seal was swimming", "The stamp’s bond failed"),
(16, "I will call you", "I will call a taxi", "Expect a phone ring from me"),
(17, "She likes cold coffee", "She likes cold weather", "Iced brew is her preference"),
(18, "He shot the film", "He shot the duck", "He directed the movie"),
(19, "The plane landed", "The plane crashed", "Touchdown occurred"),
(20, "I saved the file", "I saved the cat", "I stored the document"),
(21, "We broke the ice", "We broke the law", "We started the conversation easily"),
(22, "She drew the curtains", "She drew the sword", "She pulled the drapes shut"),
(23, "He left early", "He left dessert", "He departed ahead of time"),
(24, "They watched the stars", "They watched the stocks", "They gazed at the night sky"),
(25, "I lost my keys", "I lost my patience", "I can’t find the keyring"),
(26, "She fixed the date", "She fixed the chair", "She set the day"),
(27, "He drove to work", "He drove the workers", "He commuted by car"),
(28, "The ship sailed", "The ship sank", "The vessel departed"),
(29, "I caught the train", "I caught the trend", "I boarded the commuter rail"),
(30, "She read the letter", "She read the meter", "She reviewed the note"),
(31, "They built a house", "They built a case", "They constructed a home"),
(32, "He broke the bank", "He broke the bench", "He spent too much money"),
(33, "The leaves fell", "The leaves changed", "Foliage dropped"),
(34, "She passed the test", "She passed the salt", "She succeeded in the exam"),
(35, "The pen is blue", "The pen is mightier", "Its ink colour is azure"),
(36, "I broke my fast", "I broke my phone", "I had breakfast"),
(37, "She drew a blank", "She drew a portrait", "She couldn’t recall anything"),
(38, "He made the cut", "He made the cake", "He qualified"),
(39, "We ran the numbers", "We ran the marathon", "We did the calculations"),
(40, "The court is in session", "The court is in the park", "The judge has convened"),
(41, "I left my book", "I left my job", "I forgot the thing I was reading"),
(42, "She addressed the crowd", "She addressed the envelope", "She gave a speech"),
(43, "He met his match", "He met his manager", "He found an equal opponent"),
(44, "They saw the light", "They saw the lighthouse", "They realised the truth"),
(45, "The window is stuck", "The window is open", "It won’t slide"),
(46, "I broke the news", "I broke the vase", "I announced it first"),
(47, "She runs the show", "She runs the shop", "She’s in charge"),
(48, "He watched the clouds", "He watched the crowd", "He looked at the sky"),
(49, "We opened the case", "We opened the café", "We started the investigation"),
(50, "The plant is dying", "The plant is hiring", "The flower is withering"),
(51, "I changed my mind", "I changed my shirt", "I reconsidered"),
(52, "She closed the deal", "She closed the door", "She finalised the agreement"),
(53, "He hit the sack", "He hit the ball", "He went to sleep"),
(54, "They cracked the code", "They cracked the plate", "They solved the cipher"),
(55, "The battery died", "The battery swelled", "It ran out of power"),
(56, "I missed your call", "I missed your car", "I didn’t answer when you rang"),
(57, "She threw a party", "She threw a stone", "She hosted an event"),
(58, "He drew the line", "He drew the lion", "He set a limit"),
(59, "We signed the paper", "We signed the player", "We put signatures on the document"),
(60, "The price dropped", "The price rose", "It became cheaper"),
(61, "I caught your name", "I caught your bag", "I heard what you’re called"),
(62, "He broke his word", "He broke his sword", "He failed to keep his promise"),
(63, "They raised the bar", "They raised the barn", "They increased the standard"),
(64, "I lost the game", "I lost the luggage", "I was defeated"),
(65, "She touched the screen", "She touched the sky", "She tapped the display"),
(66, "He cleared the table", "He cleared the exam", "He removed the dishes"),
(67, "I broke the habit", "I broke the hammer", "I stopped the routine"),
(68, "I sealed the deal", "I sealed the leak", "I finalised the agreement"),
(69, "She trained the dog", "She trained the lens", "She taught the puppy"),
(70, "He nailed the presentation", "He nailed the plank", "He delivered an excellent talk"),
(71, "They fired the coach", "They fired the kiln", "They dismissed the manager"),
(72, "The star was bright", "The star was right", "It shone intensely"),
(73, "I booked an appointment", "I booked an apartment", "I scheduled a meeting"),
(74, "She filed a complaint", "She filed her nails", "She submitted a grievance"),
(75, "He secured the bag", "He secured the boat", "He obtained the money"),
(76, "We hit the road", "We hit the rock", "We set off"),
(77, "The chair is light", "The chair is heavy", "It weighs very little"),
(78, "I shot an email", "I shot an eagle", "I sent a message"),
(79, "She delivered a baby", "She delivered a pizza", "She gave birth"),
(80, "He addressed the issue", "He addressed the tissue", "He dealt with the problem"),
(81, "They charged ahead", "They charged a fee", "They moved forward quickly"),
(82, "The bridge collapsed", "The bridge expanded", "The span fell down"),
(83, "I drew a conclusion", "I drew a cube", "I inferred the result"),
(84, "She nailed the solo", "She nailed the sign", "Her performance was flawless"),
(85, "He planted the idea", "He planted the seeds", "He suggested it subtly"),
(86, "We cracked a joke", "We cracked a yolk", "We made a quip"),
(87, "The file is corrupt", "The file is correct", "The data is damaged"),
(88, "She broke ground", "She broke granite", "She started the project"),
(89, "They launched the product", "They launched the rocket", "They released the new item"),
(90, "The cat is out", "The cat is ours", "The secret has been revealed"),
(91, "I ran out of time", "I ran out of thyme", "The deadline arrived before I finished"),
(92, "She broke into tears", "She broke into the house", "She started crying"),
(93, "He pitched a tent", "He pitched an idea", "He set up a shelter"),
(94, "We drew up plans", "We drew up chairs", "We prepared blueprints"),
(95, "The bill passed", "The bill failed", "Parliament approved the law"),
(96, "I left the lights on", "I left the lights off", "I forgot to switch them off"),
(97, "She moved the goalposts", "She moved the sofa", "She unfairly changed the rules"),
(98, "He fixed the game", "He fixed the gate", "He rigged the match"),
(99, "They broke the silence", "They broke the ceiling", "Someone finally spoke"),
(100, "The deal fell through", "The deal went through", "The agreement collapsed at the last minute"),
(101, "The fact that Rango is a reptile makes the situation even worse.", "The news that Rango was a reptilian made the situation even worse.", "Rango is a reptile. What could be even worse")
]


In [14]:
tie_wrong_count = 0
tie_right_count = 0
lose_count = 0
win_count = 0

for i, s1, s2, s3 in data:
    aaa = embedding_model.encode([s1, s2, s3])['dense_vecs']

    full_right = np.dot(aaa[0],aaa[1]) < np.dot(aaa[0],aaa[2])
    
    p_1 = np.dot(
        aaa[0][list(semantic_dims)],
        aaa[1][list(semantic_dims)],
    )
    p_2 = np.dot(
        aaa[0][list(semantic_dims)],
        aaa[2][list(semantic_dims)],
    )
    partial_right = p_1 < p_2

    if((full_right and partial_right)):
        tie_right_count += 1
    elif(not full_right and not partial_right):
        tie_wrong_count += 1
    elif(not full_right and partial_right):
        # print(f'win: {i}')
        win_count += 1
    else:
        # print(f'lose: {i}')
        lose_count += 1

print(tie_wrong_count)
print(tie_right_count)
print(lose_count)
print(win_count)

16
43
20
22


In [15]:
semantic_dims

{376, 386}

In [10]:
# Set 1: Paraphrasing
set1_paraphrasing = [
    "The new regulations are expected to significantly impact the industry.",
    "It is anticipated that the new rules will have a major effect on the sector.",
    "The industry will likely change due to the new regulations.",
    "The government has introduced new regulations.",
    "The industry has been growing steadily for the past five years."
]

# Set 2: Entailment & Contradiction
set2_entailment_contradiction = [
    "All mammals breathe oxygen to survive.",
    "A dolphin, which is a mammal, needs oxygen.",
    "Many animals require oxygen.",
    "Plants produce oxygen through photosynthesis.",
    "There are mammals that do not need oxygen."
]

# Set 3: Same Topic, Different Details
set3_same_topic_different_details = [
    "The new sci-fi movie received overwhelmingly positive reviews from critics.",
    "Critics praised the new science fiction film for its stunning visuals and compelling story.",
    "The sci-fi movie was released last Friday.",
    "Many viewers found the plot of the new sci-fi movie to be confusing.",
    "The documentary about marine life won several awards."
]

# Set 4: Similar Structure, Different Key Words
set4_similar_structure_different_keywords = [
    "The team celebrated their victory after the final match.",
    "After the last game, the team rejoiced in their win.",
    "The team prepared their strategy before the final match.",
    "The chef prepared the ingredients before the dinner service.",
    "The student submitted their application after the final exam."
]

# Set 5: Abstract & Idiomatic Language
set5_abstract_idiomatic_language = [
    "The project was a walk in the park for the experienced team.",
    "The experienced team completed the project with great ease.",
    "The team found the project to be very straightforward.",
    "The project had a very tight deadline.",
    "The team went for a walk in the park after work."
]

In [11]:
random_dims = random.sample(range(1024), len(semantic_dims))


In [11]:
test = [set1_paraphrasing, set2_entailment_contradiction, set3_same_topic_different_details,set4_similar_structure_different_keywords,set5_abstract_idiomatic_language]
for set in test:
    a = embedding_model.encode(set)['dense_vecs']
    print("FULL EMBEDDING=============================")
    print(np.dot(a[0],a[1]))
    print(np.dot(a[0],a[2]))
    print(np.dot(a[0],a[3]))
    print(np.dot(a[0],a[4]))
    print("PRUNED EMBEDDING============================")
    print(np.dot(a[0][list(semantic_dims)],a[1][list(semantic_dims)]))
    print(np.dot(a[0][list(semantic_dims)],a[2][list(semantic_dims)]))
    print(np.dot(a[0][list(semantic_dims)],a[3][list(semantic_dims)]))
    print(np.dot(a[0][list(semantic_dims)],a[4][list(semantic_dims)]))
    print("RANDOM EMBEDDING============================")
    print(np.dot(a[0][list(random_dims)],a[1][list(random_dims)]))
    print(np.dot(a[0][list(random_dims)],a[2][list(random_dims)]))
    print(np.dot(a[0][list(random_dims)],a[3][list(random_dims)]))
    print(np.dot(a[0][list(random_dims)],a[4][list(random_dims)]))
    print("++++++++++++++++++++++++++++++++++++++++++++++++++++++")


FULL EMBEDDING=============================
0.9375
0.9214
0.759
0.6177
PRUNED EMBEDDING============================
0.1273
0.1301
0.1415
0.1353
RANDOM EMBEDDING============================
0.006287
0.006123
0.004383
0.00171
++++++++++++++++++++++++++++++++++++++++++++++++++++++
FULL EMBEDDING=============================
0.7593
0.802
0.679
0.7227
PRUNED EMBEDDING============================
0.1178
0.12195
0.10516
0.10956
RANDOM EMBEDDING============================
0.00736
0.005356
0.00734
0.007397
++++++++++++++++++++++++++++++++++++++++++++++++++++++
FULL EMBEDDING=============================
0.872
0.767
0.67
0.554
PRUNED EMBEDDING============================
0.11304
0.11957
0.10254
0.1296
RANDOM EMBEDDING============================
0.00408
0.004314
0.004692
0.004272
++++++++++++++++++++++++++++++++++++++++++++++++++++++
FULL EMBEDDING=============================
0.948
0.7925
0.5366
0.642
PRUNED EMBEDDING============================
0.1294
0.1254
0.11865
0.1354
RANDOM EMBEDDING===

# Synonyms

In [ ]:
from collections import Counter
syntax_dims = []


for j in range(hadamard.shape[0]):
    [syntax_dims.append(x.item()) for x in list(np.where(hadamard[j] > 0.005)[0])]


syntax_dims_list = list(syntax_dims)
print(len(syntax_dims_list))
dim_counter = Counter(syntax_dims_list)

syntax_dims_list = [x for x in dim_counter.keys() if dim_counter[x] > 15]
print(len(syntax_dims_list))
# meaning1_embedding_filtered = np.delete(meaning1_embedding, syntax_dims_list, axis=1)
# meaning2_embedding_filtered = np.delete(meaning2_embedding, syntax_dims_list, axis=1)
# def1_embedding_filtered = np.delete(def1_embedding, syntax_dims_list, axis=1)


meaning1_embedding_filtered = meaning1_embedding[:, syntax_dims_list]
meaning2_embedding_filtered = meaning1_embedding[:, syntax_dims_list]
def1_embedding_filtered = meaning1_embedding[:, syntax_dims_list]

if(np.mean((meaning1_embedding_filtered @ meaning2_embedding_filtered.T).diagonal()) < np.mean((meaning1_embedding_filtered @ def1_embedding_filtered.T).diagonal())): 
    print(np.mean((meaning1_embedding_filtered @ meaning2_embedding_filtered.T).diagonal()))
    print(np.mean((meaning1_embedding_filtered @ def1_embedding_filtered.T).diagonal()))
    